# 从零实现 MMoE 多任务推荐：多专家、多门控与缺失标签

推荐/广告系统常同时预测点击、转化、停留或满意度。完全共享容易负迁移，完全分塔又失去数据共享。MMoE 为每个任务学习独立 gate，在共享 expert 集合上形成不同加权组合。本 Notebook 手写 expert、task gate、tower、masked multi-task loss、时间切分、评估与发布制品。

合成二分类任务刻意让一个共享特征对两个任务方向相反。受控 AUC 只验证代码能学习该规则，不代表真实 CTR/CVR，因为真实系统还面对曝光选择偏差、延迟转化、校准和策略反馈回路。

In [ ]:
import copy
import hashlib
import io
import json
import math
import random
import warnings
from types import MappingProxyType

warnings.filterwarnings("ignore", message="The pynvml package is deprecated")
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED54 = 5401
random.seed(SEED54); np.random.seed(SEED54); torch.manual_seed(SEED54)
torch.set_num_threads(1)
DEVICE54 = torch.device("cpu")

def canonical_json54(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def sha54(value):
    return hashlib.sha256(value).hexdigest()

assert DEVICE54.type == "cpu" and torch.get_num_threads() == 1

## 1. 数据生成、时间切分与缺失标签

输入 `x:[N,6]`。任务 click 的 logit 包含 `+2.4*x0`，任务 conversion 包含 `-2.2*x0`，同时各有不同非线性/私有特征。这让盲目共享有潜在冲突。标签由 Bernoulli 采样；conversion 每三个样本缺一个标签，用 `label_mask:[N,2]` 表示，而不是把缺失编码成负例。

样本按模拟时间顺序切为 train/validation/test=`720/160/160`，normalizer 仅由 train 推导。这里只是协议演示；真实数据要按用户/活动/时间去重，并处理点击后才能观测转化的 censoring。

In [ ]:
FEATURE_NAMES54 = ["shared_intent", "click_context", "interaction_a", "conversion_context", "price_signal", "history_signal"]
TASK_NAMES54 = ["click", "conversion"]

def generate_events54(count=1040, seed=SEED54 + 1):
    if count < 100:
        raise ValueError("event_count_too_small")
    generator = torch.Generator().manual_seed(seed)
    x = torch.randn(count, 6, generator=generator)
    time = torch.arange(count, dtype=torch.long)
    click_logit = 2.4*x[:,0] + 1.2*x[:,1] + 1.1*x[:,1]*x[:,2] - 0.5*x[:,4]
    conversion_logit = -2.2*x[:,0] + 1.8*x[:,3] + 0.9*x[:,4]*x[:,5] + 0.3*x[:,2]
    probs = torch.stack([click_logit.sigmoid(), conversion_logit.sigmoid()], -1)
    labels = (torch.rand(count, 2, generator=generator) < probs).float()
    label_mask = torch.ones(count, 2, dtype=torch.bool)
    label_mask[::3, 1] = False
    return x, labels, label_mask, time

raw_x54, labels54, label_mask54, time54 = generate_events54()
split54 = {"train": torch.arange(0,720), "val": torch.arange(720,880), "test": torch.arange(880,1040)}
mean54 = raw_x54[split54["train"]].mean(0); std54 = raw_x54[split54["train"]].std(0, unbiased=False).clamp_min(1e-6)
x54 = (raw_x54 - mean54) / std54
assert x54.shape == (1040, 6) and labels54.shape == label_mask54.shape == (1040, 2)
assert torch.allclose(x54[split54["train"]].mean(0), torch.zeros(6), atol=1e-6)
assert time54[split54["train"]].max() < time54[split54["val"]].min() < time54[split54["test"]].min()
assert label_mask54[:,0].all() and 0 < label_mask54[:,1].sum() < len(label_mask54)
regenerated54 = generate_events54()
assert all(torch.equal(a,b) for a,b in zip(regenerated54,(raw_x54,labels54,label_mask54,time54)))

## 2. Expert、task gate 与 tower

`E` 个 expert 分别产生 `expert_outputs:[B,E,H]`。每个任务 $k$ 有独立 gate：

$$g_k(x)=\mathrm{softmax}(W_kx),\quad h_k=\sum_{e=1}^{E}g_{k,e}(x)f_e(x).$$

task tower 再把 `h_k:[B,H]` 映射为一个 logit。softmax 必须沿 expert 维，且每行和为 1。gate 不等于“把专家随机路由掉”：本例是 dense mixture，所有专家都有梯度；大规模 sparse MoE 才需要 capacity/drop/load-balance 合同。

In [ ]:
class Expert54(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
    def forward(self, x):
        return self.net(x)

class TaskTower54(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

class MMoE54(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=20, num_experts=4, num_tasks=2):
        super().__init__()
        if input_dim < 1 or hidden_dim < 2 or num_experts < 2 or num_tasks < 2:
            raise ValueError("invalid_mmoe_config")
        self.input_dim, self.hidden_dim, self.num_experts, self.num_tasks = input_dim, hidden_dim, num_experts, num_tasks
        self.experts = nn.ModuleList([Expert54(input_dim, hidden_dim) for _ in range(num_experts)])
        self.gates = nn.ModuleList([nn.Linear(input_dim, num_experts) for _ in range(num_tasks)])
        self.towers = nn.ModuleList([TaskTower54(hidden_dim) for _ in range(num_tasks)])

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.input_dim:
            raise ValueError("features_must_have_shape_B_input_dim")
        if not torch.isfinite(x).all():
            raise ValueError("nonfinite_features")
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)
        gate_weights = torch.stack([gate(x).softmax(-1) for gate in self.gates], dim=1)
        mixed = torch.einsum("bte,beh->bth", gate_weights, expert_outputs)
        logits = torch.stack([self.towers[t](mixed[:,t]) for t in range(self.num_tasks)], dim=1)
        return logits, gate_weights, expert_outputs

probe_model54 = MMoE54()
probe_logits54, probe_gates54, probe_experts54 = probe_model54(x54[:7])
assert probe_logits54.shape == (7,2) and probe_gates54.shape == (7,2,4) and probe_experts54.shape == (7,4,20)
assert torch.allclose(probe_gates54.sum(-1), torch.ones(7,2), atol=1e-7)
manual_mixed54 = sum(probe_gates54[:,0,e,None] * probe_experts54[:,e] for e in range(4))
assert torch.allclose(manual_mixed54, torch.einsum("be,beh->bh", probe_gates54[:,0], probe_experts54), atol=1e-7)
try:
    probe_model54(torch.zeros(3,5)); raise AssertionError("wrong feature shape accepted")
except ValueError as error:
    assert str(error) == "features_must_have_shape_B_input_dim"

## 3. Expert 顺序不应改变函数

Expert 是一个集合：若同时置换 `expert_outputs` 的 expert 轴与 gate 的对应列，mixed representation 必须完全相同。这个 oracle 会抓到 `einsum` 轴写反或对 task 维做 softmax。

不同任务拥有不同 gate 参数，因此可对同一输入形成不同 mixture；但 gate 可视化只是诊断，不等于因果解释或“专家学会了业务概念”。

In [ ]:
def mix_experts54(expert_outputs, gate_weights):
    if expert_outputs.ndim != 3 or gate_weights.ndim != 3:
        raise ValueError("mix_rank_contract")
    if expert_outputs.shape[:2] != (gate_weights.shape[0], gate_weights.shape[2]):
        raise ValueError("mix_shape_contract")
    if not torch.isfinite(expert_outputs).all() or not torch.isfinite(gate_weights).all() or bool((gate_weights < 0).any()):
        raise ValueError("mix_finite_nonnegative_contract")
    if not torch.allclose(gate_weights.sum(-1), torch.ones_like(gate_weights.sum(-1)), atol=1e-6):
        raise ValueError("gate_rows_must_sum_to_one")
    return torch.einsum("bte,beh->bth", gate_weights, expert_outputs)

mixed54 = mix_experts54(probe_experts54, probe_gates54)
permutation54 = torch.tensor([2,0,3,1])
permuted_mixed54 = mix_experts54(probe_experts54[:,permutation54], probe_gates54[:,:,permutation54])
assert torch.allclose(mixed54, permuted_mixed54, atol=1e-7)
assert not torch.allclose(probe_gates54[:,0], probe_gates54[:,1])
broken_gate54 = probe_gates54.clone(); broken_gate54[:,:,0] = 0
try:
    mix_experts54(probe_experts54, broken_gate54); raise AssertionError("unnormalized gate accepted")
except ValueError as error:
    assert str(error) == "gate_rows_must_sum_to_one"
negative_gate54 = probe_gates54.clone(); negative_gate54[0,0] = torch.tensor([-0.1,0.4,0.3,0.4])
try:
    mix_experts54(probe_experts54, negative_gate54); raise AssertionError("negative gate accepted")
except ValueError as error:
    assert str(error) == "mix_finite_nonnegative_contract"

## 4. 缺失标签的 masked multi-task BCE

对任务 $k$，只在 `label_mask[:,k]=True` 的样本上求 BCE，再用显式 task weight 合并。不能先把所有元素乘 mask 后直接 `.mean()`，因为缺失率会偷偷改变任务权重；也不能把缺失 conversion 当 0。

每个训练 batch 必须至少为每个任务提供一个标签，否则该任务的 loss 分母为零。本例使用全训练 split 更新，生产 minibatch 可用分层采样或跨 batch 累积。

In [ ]:
def masked_multitask_bce54(logits, labels, label_mask, task_weights=None):
    if logits.shape != labels.shape or logits.shape != label_mask.shape or logits.ndim != 2:
        raise ValueError("multitask_loss_shape_contract")
    if label_mask.dtype != torch.bool:
        raise TypeError("label_mask_must_be_bool")
    if not labels.is_floating_point() or labels.dtype != logits.dtype or labels.device != logits.device or label_mask.device != logits.device:
        raise TypeError("labels_must_match_logits_float_dtype_device")
    if not torch.isfinite(logits).all():
        raise ValueError("nonfinite_logits")
    observed_labels = labels[label_mask]
    if not torch.isfinite(observed_labels).all() or not torch.all((observed_labels == 0) | (observed_labels == 1)):
        raise ValueError("invalid_observed_binary_labels")
    if not label_mask.any(0).all():
        raise ValueError("every_task_needs_a_labeled_example")
    weights = torch.ones(logits.shape[1], device=logits.device) if task_weights is None else torch.as_tensor(task_weights, dtype=logits.dtype, device=logits.device)
    if weights.shape != (logits.shape[1],) or not torch.isfinite(weights).all() or not torch.all(weights > 0):
        raise ValueError("positive_task_weight_contract")
    losses=[]
    for task in range(logits.shape[1]):
        losses.append(F.binary_cross_entropy_with_logits(logits[label_mask[:,task],task], labels[label_mask[:,task],task]))
    per_task = torch.stack(losses)
    normalized_weights = weights / weights.max()
    total = (per_task * normalized_weights).sum() / normalized_weights.sum()
    if not torch.isfinite(total):
        raise ValueError("nonfinite_multitask_loss")
    return total, per_task

toy_logits54 = torch.tensor([[0.2,-0.3],[0.8,0.4],[-0.1,1.2]])
toy_labels54 = torch.tensor([[1.,0.],[1.,1.],[0.,0.]])
toy_mask54 = torch.tensor([[True,True],[True,False],[True,True]])
loss_a54, per_task54 = masked_multitask_bce54(toy_logits54,toy_labels54,toy_mask54)
changed_hidden_label54 = toy_labels54.clone(); changed_hidden_label54[1,1] = 1 - changed_hidden_label54[1,1]
loss_b54, _ = masked_multitask_bce54(toy_logits54,changed_hidden_label54,toy_mask54)
assert torch.allclose(loss_a54, loss_b54) and per_task54.shape == (2,)
nan_hidden_labels54 = toy_labels54.clone(); nan_hidden_labels54[1,1] = float("nan")
nan_hidden_loss54,_ = masked_multitask_bce54(toy_logits54,nan_hidden_labels54,toy_mask54)
assert torch.allclose(loss_a54,nan_hidden_loss54)
direct_task154 = F.binary_cross_entropy_with_logits(toy_logits54[[0,2],1],toy_labels54[[0,2],1])
assert torch.allclose(per_task54[1], direct_task154)
try:
    masked_multitask_bce54(toy_logits54,toy_labels54,torch.zeros_like(toy_mask54)); raise AssertionError("empty task accepted")
except ValueError as error:
    assert str(error) == "every_task_needs_a_labeled_example"
try:
    masked_multitask_bce54(toy_logits54,toy_labels54,toy_mask54,[float("inf"),1.]); raise AssertionError("infinite task weight accepted")
except ValueError as error:
    assert str(error) == "positive_task_weight_contract"
huge_weight_loss54,_ = masked_multitask_bce54(toy_logits54,toy_labels54,toy_mask54,[3e38,3e38])
assert torch.isfinite(huge_weight_loss54)
try:
    masked_multitask_bce54(toy_logits54,toy_labels54.long(),toy_mask54); raise AssertionError("integer BCE labels accepted")
except TypeError as error:
    assert str(error) == "labels_must_match_logits_float_dtype_device"

## 5. 分任务 AUC 与 mask

AUC 衡量随机正例得分高于随机负例的概率。这里用成对比较实现，显式处理 tie；验证/测试时仍只选择有标签样本。AUC 不依赖单一 threshold，但不反映概率校准、业务价值或曝光偏差。

空正例或空负例的 split 没有可定义 AUC，应 fail closed，不能返回 0 或 1 掩盖数据问题。

In [ ]:
def binary_auc54(scores, labels):
    if scores.ndim != 1 or labels.shape != scores.shape or not torch.isfinite(scores).all():
        raise ValueError("auc_shape_or_finite_contract")
    if not torch.isfinite(labels).all() or not torch.all((labels == 0) | (labels == 1)):
        raise ValueError("auc_binary_label_contract")
    positives = scores[labels == 1]; negatives = scores[labels == 0]
    if positives.numel() == 0 or negatives.numel() == 0:
        raise ValueError("auc_requires_both_classes")
    comparisons = positives[:,None] - negatives[None,:]
    return float(((comparisons > 0).float() + 0.5*(comparisons == 0).float()).mean())

assert binary_auc54(torch.tensor([0.9,0.1]),torch.tensor([1.,0.])) == 1.0
assert binary_auc54(torch.tensor([0.5,0.5]),torch.tensor([1.,0.])) == 0.5
try:
    binary_auc54(torch.tensor([0.1,0.2]),torch.ones(2)); raise AssertionError("single-class AUC accepted")
except ValueError as error:
    assert str(error) == "auc_requires_both_classes"
try:
    binary_auc54(torch.tensor([0.1,0.2]),torch.tensor([0.,2.])); raise AssertionError("non-binary AUC label accepted")
except ValueError as error:
    assert str(error) == "auc_binary_label_contract"

## 6. 受控训练与一次性测试

训练只看前 720 条，validation 用于观察，test 在训练完成后一次性计算。为了让结果可复现，本例使用 full-batch Adam；真实大数据会使用按任务标签可用性分层的 minibatch、样本权重、延迟标签修正和分布式输入管道。

两任务 loss 等权并不等于业务价值等权。生产中 task weight 应由离线/在线目标、梯度尺度、Pareto 取舍和安全约束确定并版本化。

In [ ]:
torch.manual_seed(SEED54)
model54 = MMoE54().to(DEVICE54)
optimizer54 = torch.optim.Adam(model54.parameters(), lr=0.012, weight_decay=1e-4)
train_idx54 = split54["train"]
with torch.no_grad(): initial_logits54 = model54(x54[train_idx54])[0]
initial_loss54 = float(masked_multitask_bce54(initial_logits54, labels54[train_idx54], label_mask54[train_idx54])[0])
history54=[]
for step54 in range(220):
    logits54, gates54, experts54 = model54(x54[train_idx54])
    loss54, task_losses54 = masked_multitask_bce54(logits54, labels54[train_idx54], label_mask54[train_idx54])
    optimizer54.zero_grad(set_to_none=True); loss54.backward()
    torch.nn.utils.clip_grad_norm_(model54.parameters(), 5.0); optimizer54.step()
    if step54 % 40 == 0: history54.append(float(loss54.detach()))
with torch.no_grad(): final_train_logits54 = model54(x54[train_idx54])[0]
final_loss54 = float(masked_multitask_bce54(final_train_logits54, labels54[train_idx54], label_mask54[train_idx54])[0])
assert final_loss54 < initial_loss54 - 0.12
assert all(math.isfinite(value) for value in history54)
assert all(any(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in expert.parameters()) for expert in model54.experts)
print({"initial_loss": round(initial_loss54,4), "final_loss": round(final_loss54,4)})

## 7. 验证、测试与 gate 诊断

分任务计算 validation/test AUC，并报告平均 gate entropy 观察是否极端塌缩。高 entropy 不一定好，低 entropy 也不一定坏；关键是离线指标、校准、切片稳定性和在线约束。

真实多任务推荐还要看 PR-AUC、logloss、ECE、用户/物品冷启动、长尾、群体公平、延迟转化、反事实/IPS、策略反馈以及线上主指标与护栏。

In [ ]:
def evaluate_split54(indices):
    with torch.no_grad(): logits, gates, _ = model54(x54[indices]); scores = logits.sigmoid()
    aucs=[]
    for task in range(2):
        valid = label_mask54[indices,task]
        aucs.append(binary_auc54(scores[valid,task], labels54[indices][valid,task]))
    entropy = float((-(gates * gates.clamp_min(1e-9).log()).sum(-1)).mean())
    return aucs, entropy

val_aucs54, val_gate_entropy54 = evaluate_split54(split54["val"])
test_aucs54, test_gate_entropy54 = evaluate_split54(split54["test"])
assert min(val_aucs54) > 0.76 and min(test_aucs54) > 0.76
assert 0 < val_gate_entropy54 <= math.log(4) + 1e-6
assert 0 < test_gate_entropy54 <= math.log(4) + 1e-6
with torch.no_grad():
    _, final_gates54, _ = model54(x54[:32])
assert not torch.allclose(final_gates54[:,0], final_gates54[:,1])
print({"val_auc": [round(v,4) for v in val_aucs54], "test_auc": [round(v,4) for v in test_aucs54]})

## 8. 可信发布：任务语义也是模型的一部分

manifest 绑定 feature 顺序、task/label 顺序、缺失标签规则、时间切分、train-only normalizer、模型 config、task weights 和训练 recipe。若交换 task 名称而不换 tower 行，数值仍能运行但业务语义完全错误。

state digest 纳入 key/dtype/shape/bytes；package 外的只读 registry 保存发布 bundle。loader 重新生成事件与 split、推导 normalizer，并返回 `PublishedMMoE54`：调用方必须传 `{feature_name: batch_column}`，wrapper 按发布顺序组装、标准化，再按 task 名返回概率。这样字典插入顺序不会影响结果，缺列/多列会 fail closed。整体替换模型或交换任务后重算内部 hash 都会被拒绝。

In [ ]:
def tensor_hash54(tensor):
    value=tensor.detach().cpu().contiguous()
    return sha54(str(value.dtype).encode()+canonical_json54(list(value.shape)).encode()+value.numpy().tobytes())

def state_digest54(state):
    digest=hashlib.sha256()
    for key,tensor in sorted(state.items()): digest.update(key.encode()); digest.update(tensor_hash54(tensor).encode())
    return digest.hexdigest()

data_snapshot54 = sha54("".join(tensor_hash54(v) for v in (raw_x54,labels54,label_mask54,time54)).encode())
manifest54 = {
    "artifact_id":"mmoe-two-task-v1","version":1,
    "model_config":{"input_dim":6,"hidden_dim":20,"num_experts":4,"num_tasks":2},
    "feature_names":FEATURE_NAMES54,"task_names":TASK_NAMES54,"label_map":{"0":"negative","1":"positive"},
    "data":{"count":1040,"seed":SEED54+1,"snapshot":data_snapshot54,"splits":{"train":[0,720],"val":[720,880],"test":[880,1040]},"conversion_missing_rule":"index_mod_3_eq_0"},
    "preprocess":{"kind":"train_only_standardize","mean":mean54.tolist(),"std":std54.tolist()},
    "training":{"seed":SEED54,"optimizer":"Adam","steps":220,"lr":0.012,"weight_decay":1e-4,"grad_clip_norm":5.0,"task_weights":[1.0,1.0]},
}

def build_package54(model,manifest):
    buffer=io.BytesIO(); torch.save(model.state_dict(),buffer); raw=buffer.getvalue()
    state=torch.load(io.BytesIO(raw),map_location="cpu",weights_only=True)
    manifest_sha=sha54(canonical_json54(manifest).encode()); semantic=state_digest54(state); raw_sha=sha54(raw)
    bundle=sha54(canonical_json54({"manifest_sha":manifest_sha,"state_digest":semantic,"state_bytes_sha":raw_sha}).encode())
    return {"manifest":copy.deepcopy(manifest),"manifest_sha":manifest_sha,"state_bytes":raw,"state_digest":semantic,"state_bytes_sha":raw_sha,"bundle_digest":bundle}

package54=build_package54(model54,manifest54)
PUBLISHER_REGISTRY54=MappingProxyType({("mmoe-two-task-v1",1):package54["bundle_digest"]})

class PublishedMMoE54:
    def __init__(self,model,manifest,mean,std):
        self._model=model
        self.feature_names=tuple(manifest["feature_names"])
        self.task_names=tuple(manifest["task_names"])
        self._mean=mean.detach().clone(); self._std=std.detach().clone()
    @property
    def mean(self): return self._mean.clone()
    @property
    def std(self): return self._std.clone()
    @torch.no_grad()
    def predict(self,features_by_name):
        if not isinstance(features_by_name,dict) or set(features_by_name)!=set(self.feature_names):
            raise ValueError("named_feature_schema_mismatch")
        columns=[]; batch_size=None
        for name in self.feature_names:
            value=torch.as_tensor(features_by_name[name],dtype=self._mean.dtype)
            if value.ndim!=1 or not torch.isfinite(value).all(): raise ValueError("named_feature_value_contract")
            batch_size=value.numel() if batch_size is None else batch_size
            if value.numel()!=batch_size: raise ValueError("named_feature_batch_mismatch")
            columns.append(value)
        raw=torch.stack(columns,-1); normalized=(raw-self._mean)/self._std
        logits=self._model(normalized)[0]; probabilities=logits.sigmoid()
        return MappingProxyType({name:probabilities[:,index].clone() for index,name in enumerate(self.task_names)})

def load_mmoe54(package):
    manifest=package["manifest"]; key=(manifest.get("artifact_id"),manifest.get("version"))
    if PUBLISHER_REGISTRY54.get(key)!=package.get("bundle_digest"): raise RuntimeError("publisher_registry_rejected_bundle")
    if manifest!=manifest54 or sha54(canonical_json54(manifest).encode())!=package["manifest_sha"]: raise RuntimeError("manifest_contract_mismatch")
    regenerated=generate_events54(manifest["data"]["count"],manifest["data"]["seed"])
    if sha54("".join(tensor_hash54(v) for v in regenerated).encode())!=manifest["data"]["snapshot"]: raise RuntimeError("event_snapshot_mismatch")
    regenerated_x=regenerated[0]; train_slice=slice(*manifest["data"]["splits"]["train"])
    derived_mean=regenerated_x[train_slice].mean(0); derived_std=regenerated_x[train_slice].std(0,unbiased=False).clamp_min(1e-6)
    if not torch.allclose(derived_mean,torch.tensor(manifest["preprocess"]["mean"])) or not torch.allclose(derived_std,torch.tensor(manifest["preprocess"]["std"])): raise RuntimeError("preprocess_contract_mismatch")
    if sha54(package["state_bytes"])!=package["state_bytes_sha"]: raise RuntimeError("state_bytes_mismatch")
    state=torch.load(io.BytesIO(package["state_bytes"]),map_location="cpu",weights_only=True)
    if state_digest54(state)!=package["state_digest"]: raise RuntimeError("state_semantic_mismatch")
    expected=sha54(canonical_json54({"manifest_sha":package["manifest_sha"],"state_digest":package["state_digest"],"state_bytes_sha":package["state_bytes_sha"]}).encode())
    if expected!=package["bundle_digest"]: raise RuntimeError("bundle_digest_mismatch")
    restored=MMoE54(**manifest["model_config"]); restored.load_state_dict(state); restored.eval()
    return PublishedMMoE54(restored,manifest,derived_mean,derived_std)

restored54=load_mmoe54(package54)
named_features54={name:raw_x54[:5,index] for index,name in enumerate(FEATURE_NAMES54)}
restored_predictions54=restored54.predict(named_features54)
with torch.no_grad(): direct_probabilities54=model54(x54[:5])[0].sigmoid()
assert all(torch.allclose(restored_predictions54[name],direct_probabilities54[:,index],atol=1e-7) for index,name in enumerate(TASK_NAMES54))
reversed_features54={name:named_features54[name] for name in reversed(FEATURE_NAMES54)}
reversed_predictions54=restored54.predict(reversed_features54)
assert all(torch.equal(restored_predictions54[name],reversed_predictions54[name]) for name in TASK_NAMES54)
leaked_std54=restored54.std; leaked_std54.zero_(); assert torch.all(restored54.std>0)
try:
    restored54.predict({name:value for name,value in named_features54.items() if name!=FEATURE_NAMES54[0]}); raise AssertionError("missing named feature accepted")
except ValueError as error:
    assert str(error)=="named_feature_schema_mismatch"
forged_manifest54=copy.deepcopy(manifest54); forged_manifest54["task_names"]=["conversion","click"]
forged_package54=build_package54(model54,forged_manifest54)
try:
    load_mmoe54(forged_package54); raise AssertionError("re-signed task swap accepted")
except RuntimeError as error:
    assert str(error)=="publisher_registry_rejected_bundle"
assert isinstance(PUBLISHER_REGISTRY54,MappingProxyType)

## 9. 复杂度、失败模式与原始来源

Dense MMoE expert 计算约为 $O(BE\cdot C_{expert})$，gate/tower 通常较小。常见失败包括：softmax 维度错、缺失标签当负例、task loss 用总 batch 分母、交换 task 顺序、全数据标准化、随机切分泄漏用户/时间、只看平均指标掩盖任务或人群退化。

- Ma et al., [Modeling Task Relationships in Multi-task Learning with Multi-gate Mixture-of-Experts](https://doi.org/10.1145/3219819.3220007), KDD 2018。
- Tang et al., [Progressive Layered Extraction (PLE)](https://arxiv.org/abs/2007.14722), RecSys 2020。
- Caruana, [Multitask Learning](https://link.springer.com/article/10.1023/A:1007379606734), Machine Learning 1997。

本例只验证两个合成任务和 dense gates，不覆盖海量稀疏特征、延迟反馈、校准、因果目标或在线实验。